# Running the SPARK pipeline on the *obama* video

This notebook demonstrates the full **SPARK** face-capture pipeline end-to-end on a set of
"obama" video clips, in three clearly separated steps:

1. **Preprocess the video** with [MonoFaceCompute](https://github.com/KelianB/MonoFaceCompute)
   &rarr; produces a `face_processed/` dataset (cropped frames, masks, semantic maps, landmarks,
   per-frame FLAME tracking).
2. **MultiFLARE** &rarr; reconstructs a personalized 3D face avatar (neural geometry deformer +
   shader) from the preprocessed multi-sequence data.
3. **TrackerAdaptation (EMOCA)** &rarr; adapts a pretrained EMOCA face tracker to the avatar via a
   short personalization finetune, so the tracker outputs the personalized FLAME parameters.
   We finish by rendering a tracking-overlay video and computing evaluation metrics.

### How this notebook executes things

The preprocessing repo and SPARK use **two different conda environments** (different Python
versions), so a single kernel cannot import both. Every step is therefore launched as a
subprocess via `conda run -n <env> ...`:

| Step | conda env |
|------|-----------|
| 1. Preprocessing (`process.py`) | `MonoFaceCompute` |
| 2. MultiFLARE / 3. TrackerAdaptation | `SPARK` |

The notebook kernel itself only needs `matplotlib` / `IPython.display` for visualization, so you
can run it from any kernel that has those.

> **Running top-to-bottom executes every step.** The training cells are GPU jobs. If you just
> want to see results quickly, the repo already ships fully-trained obama artifacts under
> `output/obama/` &mdash; each long-running cell has a note telling you that you may *manually*
> skip it and reuse those. `DEMO_MODE` (below) shrinks iteration counts so a full run still
> finishes quickly.


## 0. Setup &mdash; paths, helpers, environment check

In [ ]:

import os, subprocess
from pathlib import Path
from IPython.display import Image, Video, display

# ---- Paths (edit for your machine) ----------------------------------------
SPARK_ROOT = Path("/home/masahu/projects/forked-repos/SPARK")
MONO_ROOT  = Path("/home/masahu/projects/forked-repos/MonoFaceCompute")
RAW_CLIPS  = Path("/home/masahu/data/urus-videos/obama")   # holds 1.mp4 2.mp4 4.mp4 5.mp4

# Where MonoFaceCompute will WRITE the preprocessed dataset.
# NOTE: the original obama config points at MONO_ROOT/datasets/obama/face_processed, but
# `MonoFaceCompute/datasets` is a symlink to an external drive that may be unmounted. Point this
# at any writable location; the MultiFLARE config's `input_dir` is patched to match (Step 1).
DATASET_DIR = Path("/home/masahu/data/urus-videos/obama_processed/face_processed")

# ---- conda environments ---------------------------------------------------
MONO_ENV, SPARK_ENV = "MonoFaceCompute", "SPARK"

# ---- SPARK sub-projects & outputs -----------------------------------------
MF_DIR  = SPARK_ROOT / "MultiFLARE"
TA_DIR  = SPARK_ROOT / "TrackerAdaptation"
OUT_DIR = SPARK_ROOT / "output" / "obama"          # MultiFLARE output dir (from obama.txt)
EXP_DIR = OUT_DIR / "EMOCA_MultiFLARE"              # TrackerAdaptation experiment dir

# ---- Demo mode ------------------------------------------------------------
# True  -> tiny iteration counts so the whole notebook runs in a few minutes (rough avatar).
# False -> full settings from the config files (publication-quality, much slower).
DEMO_MODE = True

MF_ITERS       = 400  if DEMO_MODE else 3000   # MultiFLARE total iterations
ADAPT_ITERS    = 200  if DEMO_MODE else 3000   # TrackerAdaptation finetune iterations
TRACKER_RESUME = ADAPT_ITERS                   # checkpoint iter to use for video/eval
print(f"DEMO_MODE={DEMO_MODE}  MF_ITERS={MF_ITERS}  ADAPT_ITERS={ADAPT_ITERS}")

In [ ]:

def run(cmd: str, env: str, cwd: Path):
    """Run a shell command inside a conda env, streaming its output live."""
    full = ["conda", "run", "--no-capture-output", "-n", env, "bash", "-lc",
            f"cd '{cwd}' && {cmd}"]
    print(f"$ ({env}) cd {cwd} && {cmd}\n", flush=True)
    return subprocess.run(full, check=True)

def show(path, width=900):
    """Display an image if it exists, otherwise note that it's missing."""
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"[missing] {path}")

def show_first(folder, n=1, width=420):
    """Display the first n PNGs found in a folder (sorted)."""
    folder = Path(folder)
    pngs = sorted(folder.glob("*.png"))
    if not pngs:
        print(f"[no PNGs in] {folder}"); return
    for p in pngs[:n]:
        display(Image(filename=str(p), width=width))

In [ ]:

# Sanity checks: conda envs present, raw clips present.
envs = subprocess.run(["conda", "env", "list"], capture_output=True, text=True).stdout
for e in (MONO_ENV, SPARK_ENV):
    assert e in envs, f"conda env '{e}' not found. Available:\n{envs}"

clips = sorted(RAW_CLIPS.glob("*.mp4"))
assert clips, f"No .mp4 clips found in {RAW_CLIPS}"
print("conda envs OK:", MONO_ENV, "+", SPARK_ENV)
print("raw obama clips:", [c.name for c in clips])

## 1. Preprocess the video (MonoFaceCompute)

MonoFaceCompute turns raw `.mp4` clips into the dataset format SPARK expects. The pipeline runs
these steps per clip:

`extract` (video&rarr;frames) &rarr; `crop` (face detect + crop) &rarr; `matte` (foreground
matte) &rarr; `segment` (semantic face parsing) &rarr; `landmarks` (FAN + MediaPipe) &rarr;
`track` (FLAME tracking with EMOCA) &rarr; `optimize` (landmark-based refinement).

For each sequence `<seq>` it writes `face_processed/<seq>/` containing cropped images, `mask/`,
`semantic/`, `landmarks_*.pt`, and `flame_params_optimized.json` (per-frame pose/expression +
camera + shape). The obama avatar trains on sequences **1, 2, 4, 5** and tests on **1**.

> **Prerequisite:** the `MonoFaceCompute` conda env must exist and its model assets must already
> be downloaded (`./download_all_assets.sh` in that repo). This notebook does not re-download them.


**Write the dataset config.** We write `obama.yaml` to the MonoFaceCompute repo root (not
under the `datasets/` symlink, which may be unmounted).

In [ ]:

yaml_text = f"""\
base_dir: {RAW_CLIPS}
output_dir: {DATASET_DIR}
tracker: EMOCA
shape_tracker: SMIRK
crop_mode: constant
crop_scale: 1.4
resize: 512
sequences:
    1: {{ source: 1.mp4 }}
    2: {{ source: 2.mp4 }}
    4: {{ source: 4.mp4 }}
    5: {{ source: 5.mp4 }}
shape_sequence: "1"
steps: ["extract", "crop", "matte", "segment", "landmarks", "track", "optimize"]
"""
yaml_path = MONO_ROOT / "obama.yaml"
yaml_path.write_text(yaml_text)
print(f"wrote {yaml_path}\n")
print(yaml_text)

**Run preprocessing.** This is the heaviest step (several minutes per clip on a GPU).
Manually skip this cell if `DATASET_DIR` is already populated.

In [ ]:

run("python process.py --dataset obama.yaml", env=MONO_ENV, cwd=MONO_ROOT)

**Inspect the output** &mdash; a cropped frame and its semantic segmentation for sequence 1.

In [ ]:

seq1 = DATASET_DIR / "1"
print("sequence 1 contents:", sorted(p.name for p in seq1.iterdir()) if seq1.exists() else "[not found]")
# cropped frame (subdir name can be 'crops' or 'image' depending on MonoFaceCompute version)
for sub in ("crops", "image", "img"):
    if (seq1 / sub).is_dir():
        print(f"frame from {sub}/:"); show_first(seq1 / sub); break
if (seq1 / "semantic").is_dir():
    print("semantic mask:"); show_first(seq1 / "semantic")

**Point MultiFLARE at this dataset.** We patch `input_dir` in
`MultiFLARE/configs/obama.txt` so Step 2 reads exactly what Step 1 produced.

In [ ]:

cfg = MF_DIR / "configs" / "obama.txt"
lines = cfg.read_text().splitlines()
patched = []
for ln in lines:
    if ln.strip().startswith("input_dir"):
        patched.append(f"input_dir = {DATASET_DIR}/")
    else:
        patched.append(ln)
cfg.write_text("\n".join(patched) + "\n")
print(f"patched input_dir in {cfg} -> {DATASET_DIR}/")

## 2. MultiFLARE &mdash; reconstruct the 3D face avatar

MultiFLARE learns a **personalized neural avatar** from the preprocessed sequences: a geometry
deformer (expression-driven mesh deformation on top of a FLAME-based template, with
subdivision/remeshing) plus a neural shader for appearance. Key fields in
`MultiFLARE/configs/obama.txt`: `input_dir`, `train_dir = ["1","2","4","5"]`, `output_dir`,
`iterations = 3000`, `remesh_iterations = [500, 1000]`, `deformer_pretrain = 2000`.

The repo already ships a fully-trained obama avatar under `output/obama/` (weights at iteration
3000, meshes, training grids), shown below.


**Existing trained avatar** &mdash; a training visualization grid and the textured neutral
mesh render from the shipped 3000-iteration model.

In [ ]:

show(OUT_DIR / "images" / "grid_3000.png")
show(OUT_DIR / "meshes" / "mesh_003000.png", width=420)

**(Optional) Train the avatar yourself.** In `DEMO_MODE` we use a small iteration count and
a shrunken pretrain/remesh schedule so it finishes quickly (the resulting avatar is rough &mdash;
for a real avatar set `DEMO_MODE = False` to use the full config). **Manually skip this cell** to
keep using the shipped 3000-iteration weights.

In [ ]:

if DEMO_MODE:
    extra = (f"--iterations {MF_ITERS} --remesh_iterations 100 200 "
             f"--deformer_pretrain 200 --deformer_warmup 50 --save_frequency {MF_ITERS}")
else:
    extra = f"--iterations {MF_ITERS}"
run(f"python train.py --config configs/obama.txt {extra}", env=SPARK_ENV, cwd=MF_DIR)

In [ ]:

# Confirm a model checkpoint exists.
weights = sorted((OUT_DIR / "network_weights").glob("shader_*.pt"))
print("network weights:", [w.name for w in weights])
show_first(OUT_DIR / "images", n=1, width=900)  # latest training grid

**(Optional) Export the neutral mesh** with an albedo texture &mdash; a standalone `.obj`
you can open in any 3D viewer.

In [ ]:

mesh_out = OUT_DIR / "exported_mesh"
run(f"python export_mesh.py --config configs/obama.txt --resume 3000 "
    f"--out_dir '{mesh_out}' --tex_type albedo", env=SPARK_ENV, cwd=MF_DIR)
print("exported:", sorted(p.name for p in mesh_out.iterdir()) if mesh_out.exists() else "[none]")

## 3. TrackerAdaptation &mdash; personalize the EMOCA tracker

We now adapt a pretrained **EMOCA** face tracker to the obama avatar through a short
personalization finetune (transfer learning). The adapted encoder learns to regress the
*personalized* FLAME parameters of our avatar instead of generic FLAME.

A few config conventions worth noting:

- The tracker config references the MultiFLARE avatar by **filename**: `multiflare = obama.txt`
  (resolved under `MultiFLARE/configs/`) + `multiflare_resume = 3000`.
- **EMOCA** is selected with `encoder = DECA` + `deca_model = EMOCA_v2_lr_mse_20` +
  `deca_cfg = cfg_spark.yaml` (this is how the repo's `example_emoca.txt` does it).
- `decoder = MultiFLARE` routes the encoder's output through our personalized avatar.


**Write the EMOCA-on-obama tracker config** (the repo ships a SMIRK obama config but not an
EMOCA one).

In [ ]:

emoca_cfg_text = """\
multiflare = obama.txt
multiflare_resume = 3000
test_dirs = ["1"]

exp_name = EMOCA_MultiFLARE

encoder = DECA
deca_model = EMOCA_v2_lr_mse_20
deca_cfg = cfg_spark.yaml

decoder = MultiFLARE

batch_size = 12
adapt_iters = 3000
adapt_lr = 1e-5
train_mlps = True
train_backbones_last = True
"""
emoca_cfg = TA_DIR / "configs" / "configs-to-run" / "example_emoca_obama.txt"
emoca_cfg.parent.mkdir(parents=True, exist_ok=True)
emoca_cfg.write_text(emoca_cfg_text)
print(f"wrote {emoca_cfg}\n")
print(emoca_cfg_text)

**Run the personalization finetune.** In `DEMO_MODE` this is a short run
(`adapt_iters` small, checkpoint + validation every `ADAPT_ITERS`). Checkpoints are written to
`output/obama/EMOCA_MultiFLARE/checkpoints/deca_iter<NNNN>.pth`.

> The repo ships pretrained EMOCA checkpoints at iter 1000/2000/3000. To reuse those instead of
> training, manually skip this cell and set `TRACKER_RESUME = 3000` in the next cells.

In [ ]:

cfg_rel = "configs/configs-to-run/example_emoca_obama.txt"
extra = (f"--adapt_iters {ADAPT_ITERS} --save_frequency {ADAPT_ITERS} "
         f"--val_frequency {ADAPT_ITERS} --test_frequency {ADAPT_ITERS}") if DEMO_MODE else ""
run(f"python train.py --config {cfg_rel} {extra}", env=SPARK_ENV, cwd=TA_DIR)

In [ ]:

ckpts = sorted((EXP_DIR / "checkpoints").glob("deca_iter*.pth"))
print("tracker checkpoints:", [c.name for c in ckpts])
# A validation visualization produced during adaptation (avatar overlay vs. input).
show_first(EXP_DIR / "val", n=1, width=900)

## 4. Results &mdash; tracking-overlay video and evaluation

Finally we run the adapted tracker over a held-out sequence and render an **overlay video** (the
tracked avatar composited onto the input frames), then compute quantitative metrics.


**Render the overlay video** on test sequence 1. `--tracker_resume` selects which checkpoint
to load (set to `TRACKER_RESUME`; use `3000` to use the shipped checkpoint). The video is written
to `output/obama/EMOCA_MultiFLARE/tracking_video/video.mp4`.

In [ ]:

cfg_rel = "configs/configs-to-run/example_emoca_obama.txt"
run(f"python make_overlay_video.py --config {cfg_rel} --tracker_resume {TRACKER_RESUME} "
    f"--test_dirs 1 --out tracking_video --n_frames 200 --smooth_crops --framerate 24",
    env=SPARK_ENV, cwd=TA_DIR)

In [ ]:

video_path = EXP_DIR / "tracking_video" / "video.mp4"
if video_path.exists():
    display(Video(str(video_path), embed=True, width=720))
else:
    print(f"[missing] {video_path}")

**(Optional) Quantitative evaluation** &mdash; warping error + landmark/semantic
reprojection metrics on the test set; results are also written to
`evaluation_warp_<resume>_test/results.txt`.

In [ ]:

cfg_rel = "configs/configs-to-run/example_emoca_obama.txt"
run(f"python evaluate.py --config {cfg_rel} --tracker_resume {TRACKER_RESUME} "
    f"--frame_interval 5 --num_frames 64", env=SPARK_ENV, cwd=TA_DIR)

res = EXP_DIR / f"evaluation_warp_{TRACKER_RESUME}_test" / "results.txt"
print("\n--- results.txt ---")
print(res.read_text() if res.exists() else f"[missing] {res}")

## Recap

We ran the full SPARK pipeline on the obama clips, producing three artifacts:

1. **Preprocessed dataset** &mdash; `face_processed/{1,2,4,5}/` (frames, masks, semantics,
   landmarks, FLAME tracking).
2. **Personalized 3D avatar** &mdash; MultiFLARE neural geometry + shader (`output/obama/`,
   exported neutral mesh).
3. **Adapted EMOCA tracker** &mdash; finetuned encoder + a tracking-overlay video and evaluation
   metrics (`output/obama/EMOCA_MultiFLARE/`).

To produce a publication-quality result rather than a quick demo, set `DEMO_MODE = False` in
Section 0 and re-run Steps 2&ndash;4 (much slower).
